# Dataset overview

## Summary
Source verification and structural profiling of UCI Online Retail. The workbook contains 541,909 invoice line items across eight columns, covering December 2010 to December 2011. Prices are GBP.

Source: Chen, D. (2015). [Online Retail, UCI](https://doi.org/10.24432/C5BW33), CC BY 4.0. Each row represents an invoice line item; distinct invoice identifiers are not equivalent to completed purchase orders.

## Environment and scope
The complete original workbook is loaded without filtering. Identifier columns are read as text. Runtime versions are recorded below; exact dependencies are retained in `requirements-lock.txt`.

In [1]:
from pathlib import Path
from hashlib import sha256
import json
import sys
import pandas as pd
from IPython.display import display

# Support launching from either the project root or the notebooks folder.
current_folder = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in [current_folder, *current_folder.parents]
     if (folder / 'scripts' / 'download_data.py').is_file()
     and (folder / 'requirements.txt').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Notebook must run within the project directory.')

DATA_FILE = PROJECT_ROOT / 'data' / 'raw' / 'Online Retail.xlsx'
pd.set_option('display.max_columns', 8)
print(f'Python: {sys.version.split()[0]} | pandas: {pd.__version__}')


Python: 3.14.6 | pandas: 3.1.1


## Source verification
The workbook checksum is compared with the ingestion manifest before loading the `Online Retail` sheet. The original file is not modified.

In [2]:
if not DATA_FILE.exists():
    raise FileNotFoundError('Run scripts/download_data.py from the project folder first.')

manifest_file = DATA_FILE.parent / 'source_manifest.json'
manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
assert sha256(DATA_FILE.read_bytes()).hexdigest() == manifest['sha256'], 'Source file changed.'

retail_raw = pd.read_excel(
    DATA_FILE,
    sheet_name='Online Retail',
    engine='openpyxl',
    dtype={'InvoiceNo': 'string', 'StockCode': 'string', 'CustomerID': 'string'},
)
print(f'Loaded {len(retail_raw):,} rows and {len(retail_raw.columns)} columns.')

Loaded 541,909 rows and 8 columns.


## Source sample
Five original rows illustrate the invoice-line grain. Repeated invoice IDs represent multiple entries on an invoice.

In [3]:
retail_raw.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


## Schema and completeness
Loaded column types, populated values, and missing-value counts are inspected against the source data dictionary.

In [4]:
column_summary = pd.DataFrame({
    'Data type': retail_raw.dtypes.astype(str),
    'Present rows': retail_raw.notna().sum(),
    'Missing rows': retail_raw.isna().sum(),
})
display(column_summary)

,Data type,Present rows,Missing rows
InvoiceNo,string,541909,0
StockCode,string,541909,0
Description,object,540455,1454
Quantity,int64,541909,0
InvoiceDate,datetime64[us],541909,0
UnitPrice,float64,541909,0
CustomerID,string,406829,135080
Country,str,541909,0


## Coverage and identifier cardinality
Counts include cancellations and adjustments. Customer cardinality excludes missing IDs. The final calendar month is partial.

In [5]:
raw_overview = pd.Series({
    'Invoice line items': len(retail_raw),
    'Distinct raw invoice IDs': retail_raw['InvoiceNo'].nunique(),
    'Distinct stock codes': retail_raw['StockCode'].nunique(),
    'Distinct known customer IDs': retail_raw['CustomerID'].nunique(),
    'First recorded timestamp': retail_raw['InvoiceDate'].min(),
    'Last recorded timestamp': retail_raw['InvoiceDate'].max(),
}, name='Raw file overview')
display(raw_overview.to_frame())

,Raw file overview
Invoice line items,541909
Distinct raw invoice IDs,25900
Distinct stock codes,4070
Distinct known customer IDs,4372
First recorded timestamp,2010-12-01 08:26:00
Last recorded timestamp,2011-12-09 12:50:00


## Validation
Assertions check the expected schema, source dimensions, and datetime parsing. The input checksum is verified during ingestion.

In [6]:
expected_columns = [
    'InvoiceNo', 'StockCode', 'Description', 'Quantity',
    'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country',
]
assert retail_raw.columns.tolist() == expected_columns, 'Unexpected columns.'
assert retail_raw.shape == (541_909, 8), 'Unexpected dataset size.'
assert pd.api.types.is_datetime64_any_dtype(retail_raw['InvoiceDate'])
print('Source checksum, dataset shape, columns, and date type checks passed.')

Source checksum, dataset shape, columns, and date type checks passed.


## Analytical implications
Customer-level coverage, cancellations, duplicate-looking rows, numeric exceptions, and product-code classifications require review before business metrics are aggregated. The accompanying data quality assessment documents those findings.